# KPSS CANAVARI - Veri Ön İşleme Pipeline

Bu notebook Google Drive'daki ~1000 KPSS kitabını işleyerek RAG (Retrieval-Augmented Generation) sistemi için hazırlayacak.

## Adımlar:
1. Google Drive'ı bağla
2. PDF'leri oku ve metin çıkar
3. Veri temizleme
4. Chunk'lara bölme
5. Metadata oluşturma
6. Checkpoint'lere kaydetme

## Gereksinimler:
- Google Colab Pro (önerilir - daha fazla RAM ve süre)
- 47 GB Google Drive alanı
- Tahmini süre: 6-10 saat

In [ ]:
# 1. GEREKLI KÜTÜPHANELERI YÜKLE
!pip install -q pypdf2 pdfplumber pdf2image pytesseract pillow tqdm langchain tiktoken

print('✅ Kütüphaneler yüklendi!')

In [ ]:
# 2. GOOGLE DRIVE'I BAĞLA
from google.colab import drive
import os

drive.mount('/content/drive')

# Drive klasör yolu
BOOKS_PATH = '/content/drive/MyDrive/KPSS_Kitaplar'  # Bu yolu kendi Drive yapınıza göre güncelleyin
OUTPUT_PATH = '/content/drive/MyDrive/KPSS_Processed'

# Output klasörünü oluştur
os.makedirs(OUTPUT_PATH, exist_ok=True)

print(f'✅ Google Drive bağlandı!')
print(f'📁 Kitaplar klasörü: {BOOKS_PATH}')
print(f'📁 İşlenmiş veri klasörü: {OUTPUT_PATH}')

In [ ]:
# 3. PDF DOSYALARINI BUL
import glob
from pathlib import Path

# Tüm PDF dosyalarını bul (recursive)
pdf_files = glob.glob(os.path.join(BOOKS_PATH, '**/*.pdf'), recursive=True)

print(f'📚 Toplam {len(pdf_files)} PDF dosyası bulundu!')
print('\nİlk 10 dosya:')
for i, file in enumerate(pdf_files[:10]):
    print(f'{i+1}. {Path(file).name}')

In [ ]:
# 4. PDF OKUMA FONKSİYONLARI
import PyPDF2
import pdfplumber
import json
from tqdm import tqdm

def extract_text_from_pdf(pdf_path):
    """
    PDF'den metin çıkarır. Önce PyPDF2, başarısız olursa pdfplumber kullanır.
    """
    text = ""
    try:
        # Yöntem 1: PyPDF2 (daha hızlı)
        with open(pdf_path, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)
            for page in pdf_reader.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
        
        # Eğer metin çıkmadıysa pdfplumber dene
        if len(text.strip()) < 100:
            text = ""
            with pdfplumber.open(pdf_path) as pdf:
                for page in pdf.pages:
                    page_text = page.extract_text()
                    if page_text:
                        text += page_text + "\n"
        
        return text
    
    except Exception as e:
        print(f'⚠️ Hata: {pdf_path} - {str(e)}')
        return ""

def clean_text(text):
    """
    Metni temizler: gereksiz boşluklar, özel karakterler vb.
    """
    import re
    
    # Birden fazla boşluğu tek boşluğa çevir
    text = re.sub(r'\s+', ' ', text)
    
    # Birden fazla satır sonu tek satır sonu
    text = re.sub(r'\n+', '\n', text)
    
    # Başta ve sonda boşlukları kaldır
    text = text.strip()
    
    return text

print('✅ PDF okuma fonksiyonları hazır!')

In [ ]:
# 5. CHECKPOINT SİSTEMİ
import pickle

CHECKPOINT_FILE = os.path.join(OUTPUT_PATH, 'processing_checkpoint.pkl')

def save_checkpoint(processed_files, extracted_data):
    """Checkpoint kaydet"""
    checkpoint = {
        'processed_files': processed_files,
        'extracted_data': extracted_data
    }
    with open(CHECKPOINT_FILE, 'wb') as f:
        pickle.dump(checkpoint, f)
    print(f'💾 Checkpoint kaydedildi: {len(processed_files)} dosya işlendi')

def load_checkpoint():
    """Checkpoint yükle"""
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, 'rb') as f:
            checkpoint = pickle.load(f)
        print(f'📥 Checkpoint yüklendi: {len(checkpoint["processed_files"])} dosya mevcut')
        return checkpoint['processed_files'], checkpoint['extracted_data']
    return set(), []

print('✅ Checkpoint sistemi hazır!')

In [ ]:
# 6. TÜM PDF'LERİ İŞLE
import time

# Checkpoint'ten devam et
processed_files, extracted_data = load_checkpoint()

print(f'🚀 İşleme başlıyor... ({len(pdf_files) - len(processed_files)} dosya kaldı)')

start_time = time.time()

for i, pdf_path in enumerate(tqdm(pdf_files, desc='PDF İşleniyor')):
    # Zaten işlendiyse atla
    if pdf_path in processed_files:
        continue
    
    try:
        # PDF'den metin çıkar
        text = extract_text_from_pdf(pdf_path)
        
        # Metin temizle
        text = clean_text(text)
        
        # Eğer yeterli metin varsa kaydet
        if len(text) > 100:
            file_name = Path(pdf_path).name
            extracted_data.append({
                'file_name': file_name,
                'file_path': pdf_path,
                'text': text,
                'length': len(text)
            })
            processed_files.add(pdf_path)
        
        # Her 50 dosyada bir checkpoint kaydet
        if (i + 1) % 50 == 0:
            save_checkpoint(processed_files, extracted_data)
            elapsed = time.time() - start_time
            print(f'⏱️ Geçen süre: {elapsed/60:.1f} dakika')
    
    except Exception as e:
        print(f'❌ Hata ({pdf_path}): {str(e)}')
        continue

# Final checkpoint
save_checkpoint(processed_files, extracted_data)

elapsed = time.time() - start_time
print(f'\n✅ İşleme tamamlandı!')
print(f'📊 Toplam işlenen dosya: {len(extracted_data)}')
print(f'⏱️ Toplam süre: {elapsed/60:.1f} dakika')
print(f'📝 Toplam metin: {sum([d["length"] for d in extracted_data]):,} karakter')

In [ ]:
# 7. METINLERI CHUNK'LARA BÖL
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Chunk ayarları
CHUNK_SIZE = 1000  # Her chunk'ta ~1000 karakter
CHUNK_OVERLAP = 200  # Chunk'lar arası 200 karakter overlap (context kaybını önler)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)

chunks = []

print('🔪 Metinler chunk\'lara bölünüyor...')

for data in tqdm(extracted_data, desc='Chunking'):
    # Metni chunk'lara böl
    text_chunks = text_splitter.split_text(data['text'])
    
    # Her chunk için metadata ekle
    for i, chunk in enumerate(text_chunks):
        chunks.append({
            'chunk_id': f"{data['file_name']}_{i}",
            'source_file': data['file_name'],
            'chunk_index': i,
            'text': chunk,
            'length': len(chunk)
        })

print(f'\n✅ Chunking tamamlandı!')
print(f'📦 Toplam chunk sayısı: {len(chunks):,}')
print(f'📏 Ortalama chunk uzunluğu: {sum([c["length"] for c in chunks]) / len(chunks):.0f} karakter')

In [ ]:
# 8. CHUNK'LARI KAYDET
import json

# JSON olarak kaydet
chunks_file = os.path.join(OUTPUT_PATH, 'chunks.json')
with open(chunks_file, 'w', encoding='utf-8') as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

print(f'💾 Chunk\'lar kaydedildi: {chunks_file}')

# İstatistikler
stats = {
    'total_files': len(extracted_data),
    'total_chunks': len(chunks),
    'total_characters': sum([c['length'] for c in chunks]),
    'avg_chunk_size': sum([c['length'] for c in chunks]) / len(chunks),
    'processing_time_minutes': elapsed / 60
}

stats_file = os.path.join(OUTPUT_PATH, 'stats.json')
with open(stats_file, 'w', encoding='utf-8') as f:
    json.dump(stats, f, indent=2)

print(f'📊 İstatistikler kaydedildi: {stats_file}')
print('\n✅ VERİ ÖN İŞLEME TAMAMLANDI!')
print('\n📋 Özet:')
for key, value in stats.items():
    print(f'  {key}: {value}')

## ✅ TAMAMLANDI!

Sonraki adım: `02_embedding_creation.ipynb` notebook'unu çalıştırarak embedding'leri oluşturun.